# Laborator 5

## IDA*

Deoarece algoritmul A* folosește o coadă, în care salvează întreaga frontieră a arborelui de căutare, se poate ajunge ușor la o cantitate mare de memorie ocupată, iar pentru unele probleme, se depășesc resursele sistemului ducând la eșecul algoritmului.

Prin urmare s-a încercat limitarea memoriei utilizate combinând A* cu modul de lucru al strategiei DepthFirst.

In [101]:
import copy
from operator import truediv


class Nod:
    def __init__(self, info, parinte=None, g=0, h=0, detalii_mutare=None):
        self.info = info
        self.parinte = parinte
        self.g = g
        self.h = h
        self.f = g + h
        self.detalii_mutare = detalii_mutare

    def obtine_drum(self):
        drum = []
        nod = self
        while nod is not None:
            drum.append(nod)
            nod = nod.parinte
        drum.reverse()
        return drum

    def in_drum(self, info):
        nod = self
        while nod is not None:
            if nod.info == info:
                return True
            nod = nod.parinte
        return False

    def __lt__(self, alt_nod):
        return self.g > alt_nod.g

    def __repr__(self):
        return f"Nod(info={self.info}, g={self.g}, h={self.h}, f={self.f})"


class Graf:
    SCOP_8PUZZLE = [[1, 2, 3], [4, 5, 6], [7, 8, 0]]

    def __init__(self, start=None, scop=None):
        self.start = start
        self.scop = scop if scop is not None else self.SCOP_8PUZZLE

    def citeste_din_fisier(self, cale_fisier):
        with open(cale_fisier, 'r') as f:
            linii = [l.strip() for l in f if l.strip()]
        self.start = [[int(x) for x in linie.split()] for linie in linii]

    def valideaza(self):
        # verifcam ca matricea sa fie de 3 pe 3
        if len(self.start)!=3:
            return False
        for rand in self.start:
            if len(rand)!=3:
                return False

        #verificam ca toate elementele sa fie de la 0 la 8
        idk=sorted([val for rand in self.start for val in rand])
        if idk!=[0,1,2,3,4,5,6,7,8]:
            return False

        # calc inversiuni :,)??? SAI cu vladoiu PTSD
        elemente_fara_zero=[val for val in self.start if val!=0]
        n=len(elemente_fara_zero)
        nr_inversiuni=0

        for i in range(0,n):
            for j in range(i+1,n):
                if elemente_fara_zero[i]>elemente_fara_zero[j]:
                    nr_inversiuni+=1

        #daca nr de inversiuni e impar, nu putem ajunge din starea init in starea scop
        if nr_inversiuni%2==1:
            return False


        return True



    def scop_func(self, nod):
        if nod == self.scop:
            return True
        else :
            return False

    def cauta(self,nod, valoare):
        for i in range(3):
            for j in range(3):
                if nod[i][j]==valoare:
                    return i,j
        return -1,-1

    def succesori(self, nod):
        lista_succesori=[]


        vector_coord=[(1,0),(-1,0),(0,1),(0,-1)] # jos, sus, dr, st

        #cautam pozitia casutei libere
        l_zero,c_zero=self.cauta(nod.info,0)

        for x,y in vector_coord:
            l_nou=l_zero+x
            c_nou=c_zero+y

            if 0<=l_nou<=2 and 0<=c_nou<=2:

                #deepcopy ca sa copiem doar valorile
                mutare_noua=copy.deepcopy(nod.info)



                val_placuta_mutata=mutare_noua[l_nou][c_nou]

                #dam swap la placuta zero cu mutarea
                mutare_noua[l_zero][c_zero]= val_placuta_mutata
                mutare_noua[l_nou][c_nou] = 0


                #verificam ca starea noua sa nu mie fie vizitata in drumul curent
                if not nod.in_drum(mutare_noua):
                    cost_mutare=1

                    detalii = f"se muta placuta {val_placuta_mutata} de pe ({l_nou},{c_nou}) pe ({l_zero},{c_zero}) cu costul {cost_mutare}."

                    h_suc=self.estimeaza_h(mutare_noua)
                    g_suc=nod.g+cost_mutare

                    nod_succesor= Nod(mutare_noua, parinte=nod, g=g_suc,h=h_suc,detalii_mutare=detalii)

                    lista_succesori.append(nod_succesor)

        return lista_succesori


    def estimeaza_h(self, nod):
        #folosim distanta manhatan
        h=0
        for l in range(3):
            for c in range(3):
                val=nod[l][c]
                if val!=0:
                    l_final,c_final=self.cauta(self.scop,val)

                    dist_menhetan= abs(l-l_final)+abs(c-c_final)
                    h+=dist_menhetan
        return h




## Problema 8-puzzle

Într-o cutie pătratică (văzută ca o matrice de dimensiune 3x3) sunt 8 plăcuțe, numerotate de la 1 la 8 și un loc liber. Plăcuțele sunt inițial amestecate. Plăcuțele pot fi mutate pe linie sau coloană doar în spațiul liber (dacă e vecin cu ele). Scopul este să ajungem cu plăcuțele ordonate crescător.

## Cerința 1 – Citire fișier, validare și funcție scop

a) Se va citi dintr-un fișier starea inițială. Starea inițială va fi reprezentată astfel: fiecare linie de plăcuțe va fi reprezentată de o linie în fișier cu numerele plăcuțelor. Locul liber va fi simbolizat prin cifra 0. Numerele vor fi separate prin câte un spațiu. O stare va fi memorată ca matrice (o listă de liste cu numerele plăcuțelor și 0 pentru locul gol).

b) Scrieți metoda `valideaza()` în clasa Graf care verifică dacă fișierul dat e valid:
- S-a citit o matrice corectă (este o matrice pătratică de dimensiune 3x3)
- Numerele din matrice sunt toate cele de la 0 la 8 inclusiv
- Dacă din starea inițială se poate ajunge într-o stare scop (numărul de inversiuni din matrice trebuie să fie par)

c) Modificați funcția `scop_func()` astfel încât să verifice că informația nodului curent este egală cu scopul cerut.

Exemple de fișiere de input:
```
2 6 4       1 0 3
7 8 1       4 2 5
5 0 3       7 8 6
```

In [102]:
input1=[[2,6,4],[7,8,1],[5,0,3]]
input2=[[1,0,3],[4,2,5],[7,8,6,]]
scop=[[1,2,3], [4,5,6], [7,8,0]]

graf= Graf(input1)
print("test validare:")
print(graf.valideaza())

print("test scop:")
print(graf.scop_func(input1))
print(graf.scop_func(scop))

test validare:
False
test scop:
False
True


## Cerința 2 – Generare succesori

Modificați funcția `succesori(nod)` astfel încât să genereze toți succesorii valizi (a căror informație nu se repetă în istoricul lor), pentru această problemă. Creați două obiecte de tip Nod cu informații diferite și afișați succesorii pentru ele, verificând corectitudinea informațiilor.


Pentru rularea cu A* cerută mai jos se consideră costul pe o mutare egal cu 1.

Gasiti o euristica admisibila si cat mai informativa pentru problema data si implementati-o in functia estimeaza_h().

In [103]:
graf_test = Graf()
nod_test1 = Nod(info=[[2,6,4], [7,8,1], [5,0,3]])
nod_test2 = Nod(info=[[1,0,3], [4,2,5], [7,8,6]])


for succesor in graf_test.succesori(nod_test1):
    print(succesor)

Nod(info=[[2, 6, 4], [7, 0, 1], [5, 8, 3]], g=1, h=14, f=15)
Nod(info=[[2, 6, 4], [7, 8, 1], [5, 3, 0]], g=1, h=16, f=17)
Nod(info=[[2, 6, 4], [7, 8, 1], [0, 5, 3]], g=1, h=14, f=15)


## Cerința 3 – Algoritmul A*

Rezolvați problema 8-puzzle folosind A* urmând cerințele:
- Considerați costul pe orice mutare egal cu numărul plăcuței care se mută.
- Implementați funcția `estimeaza_h()` corespunzătoare problemei, funcția trebuind să returneze o estimație admisibilă pentru fiecare nod. Gasiti o astfel de functie cat mai informativa si implementati-o.

In [104]:
import heapq

def A_star(graf):

    if graf.valideaza() == False:
        return None

    nod_start = Nod(graf.start, h=graf.estimeaza_h(graf.start))

    minheap = []
    #adaugam in frontiear nodurile cu f cel mai mic
    heapq.heappush(minheap,(nod_start.f, nod_start))

    noduri_vizitate = set()

    while minheap:
        f_curent,nod_curent=heapq.heappop(minheap)

       #transformam matricea in string
        if str(nod_curent.info) in noduri_vizitate:
            continue

        #ca sa evitm cicluri
        noduri_vizitate.add(str(nod_curent.info))

        if graf.scop_func(nod_curent.info):
            return nod_curent.obtine_drum()

        for succesor in graf.succesori(nod_curent):
            if str(succesor.info) not in noduri_vizitate:
                heapq.heappush(minheap, (succesor.f, succesor))

    return None





## Cerința 4 – IDA*

Implementati si IDA* pentru problema data.

In [105]:


def cautare(graf, nod, limita):

    #verificam daca nodu merita sa exploram
    if nod.f>limita:
        return False, nod.f

    #daca e scopul returnam drumu
    if graf.scop_func(nod.info):
        return True, nod.obtine_drum()




    #facem un dfs recursiv
    for succesor in graf.succesori(nod):
        gasit, cost_nou = cautare(graf, succesor, limita)

        if gasit == True:
            return True, cost_nou

        #actualizam limtia
        if cost_nou < limita_min:
            limita_min = cost_nou

    return False, limita_min



def IDA_star(graf):
    if graf.valideaza() == False:
        return None

    nod_start = Nod(graf.start, h=graf.estimeaza_h(graf.start))
    limita = nod_start.f

    while True:

        limita_min = 10000000000000000
        gasit, rezultat = cautare(graf, nod_start, limita)

        if gasit:
            return rezultat

        if rezultat == 10000000000000000:
            return None

        limita = rezultat

## Cerința 5 – Afișare drum în format matricial

Pentru problema 8-puzzle vom modifica modul de afișare astfel încât să apară în format matricial, iar numărul stării din drum să apară înainte de informația stării. Sub fiecare stare se afișează și costul total de până atunci și cât se estimează până la starea scop.

De exemplu, pentru starea inițială:
```
1)
1 0 3
4 2 5
7 8 6
cost total:0; estimat:3
```
Pentru stările de la a doua încolo se va afișa și cum s-a ajuns în starea respectivă (ce plăcuță s-a mutat, de pe ce poziție, pe ce poziție) și cu ce cost:
```
se muta placuta 2 de pe (1,1) pe (0,1) cu costul 2.
2)
1 2 3
4 0 5
7 8 6
cost total:2; estimat:2
```

In [106]:
def afiseaza_drum(drum):
    if drum is None:
        print("Nu s-a gasit niciun drum")
        return

    for i, nod in enumerate(drum):
       #afisam detaliile mutarii daca e primul nod
        if nod.detalii_mutare:
            print(nod.detalii_mutare)

        #afisam nr mutrai
        print(f"{i+1})")

        #printam matricea
        for rand in nod.info:
            print(" ".join(str(val) for val in rand))

        #printam costurile
        print(f"cost total:{nod.g}; estimat:{nod.h}")


graf_test = Graf(input2)
drum_gasit = aStar(graf_test)
afiseaza_drum(drum_gasit)

1)
1 0 3
4 2 5
7 8 6
cost total:0; estimat:3
se muta placuta 2 de pe (1,1) pe (0,1) cu costul 1.
2)
1 2 3
4 0 5
7 8 6
cost total:1; estimat:2
se muta placuta 5 de pe (1,2) pe (1,1) cu costul 1.
3)
1 2 3
4 5 0
7 8 6
cost total:2; estimat:1
se muta placuta 6 de pe (2,2) pe (1,2) cu costul 1.
4)
1 2 3
4 5 6
7 8 0
cost total:3; estimat:0
